In [1]:
!pip install requests pandas plotly duckdb streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 69.6 MB/s eta 0:00:00


In [12]:
from google.colab import userdata
import os

APP_ID  = userdata.get('ADZUNA_APP_ID')
APP_KEY = userdata.get('ADZUNA_APP_KEY')

print("Keys loaded:", bool(APP_ID), bool(APP_KEY))

Keys loaded: True True


In [13]:
import requests, pandas as pd, time
from google.colab import userdata

APP_ID  = userdata.get('ADZUNA_APP_ID')
APP_KEY = userdata.get('ADZUNA_APP_KEY')

cities = ['San Francisco', 'New York', 'Austin', 'Chicago', 'Remote']

# Search by SKILL KEYWORDS not assumed titles
# This pulls any job that mentions these in title or description
keywords = [
    'data analysis',
    'sql analytics',
    'business intelligence',
    'data reporting',
    'data insights'
]

print(f"{len(cities)} cities x {len(keywords)} keywords = {len(cities)*len(keywords)} combos")

5 cities x 5 keywords = 25 combos


In [14]:
all_jobs = []

for city in cities:
    for kw in keywords:
        for page in range(1, 4):
            try:
                r = requests.get(
                    f'https://api.adzuna.com/v1/api/jobs/us/search/{page}',
                    params={
                        'app_id': APP_ID,
                        'app_key': APP_KEY,
                        'results_per_page': 50,
                        'what': kw,
                        'where': city
                    }
                )
                results = r.json().get('results', [])
                for j in results:
                    j['_city']    = city
                    j['_keyword'] = kw
                all_jobs.extend(results)
                print(f'{city} | {kw} | p{page} → {len(all_jobs)}', end='\r')
                time.sleep(0.4)
            except Exception as e:
                print(f'Error: {e}')

print(f'\nDone: {len(all_jobs)} raw postings')



Done: 2879 raw postings


In [17]:
from collections import Counter
import re

# Deduplicate first
rows, seen = [], set()
for j in all_jobs:
    jid = j.get('id','')
    if jid in seen: continue
    seen.add(jid)
    rows.append({
        'id':           jid,
        'title':        j.get('title','').strip(),
        'company':      j.get('company',{}).get('display_name',''),
        'description':  j.get('description',''),
        'salary_min':   j.get('salary_min', 0),
        'salary_max':   j.get('salary_max', 0),
        'city':         j.get('_city',''),
        'keyword':      j.get('_keyword','')
    })

df = pd.DataFrame(rows)

# Now extract what titles companies actually use
# Normalize: lowercase, strip seniority words to group variants
def normalize_title(t):
    t = t.lower()
    for word in ['senior','sr.','sr ','junior','jr.','jr ','lead ',
                 'staff ','principal ','associate ','i ','ii ','iii ']:
        t = t.replace(word, '')
    return t.strip()

df['title_clean'] = df['title'].apply(normalize_title)

title_counts = df['title_clean'].value_counts().head(30)
print("Top 30 actual job titles companies are using:\n")
for title, count in title_counts.items():
    print(f'  {title:45} {count}')

Top 30 actual job titles companies are using:

  teacher (all experience levels) - school year 25-26 new bronx k-12 campus opening soon 139
  data engineer                                 31
  data analyst                                  24
  data scientist                                21
  product manager                               20
  business intelligence analyst                 15
  business analyst                              8
  financial analyst                             8
  account executive                             8
  strategy & planning analyst, uber advertising 6
  urologist                                     6
  applied ahealth data architect- manager       6
  software engineer                             6
  project manager                               6
  growth marketing manager                      5
  aengineer                                     5
  experience design vice president              5
  manager, strategy and operations              5
  gps

Instead of searching predefined job titles, I searched by
skill keywords inside job descriptions. This pulls any role
that requires these skills regardless of what companies
call the position.

I then extracted and normalized the actual titles that
appeared in results — stripping seniority labels to group
variants — to let the data reveal what titles exist,
not what I assumed.

This matters because a job seeker searching only "data analyst"
misses roles like "revenue analyst", "decision support analyst",
and "people analytics specialist" that require identical skills.





In [19]:
all_jobs = []

for city in cities:
    for kw in keywords:
        for page in range(1, 4):
            try:
                r = requests.get(
                    f'https://api.adzuna.com/v1/api/jobs/us/search/{page}',
                    params={
                        'app_id': APP_ID,
                        'app_key': APP_KEY,
                        'results_per_page': 50,
                        'what': kw,
                        'where': city
                    }
                )
                results = r.json().get('results', [])
                for j in results:
                    j['_city']    = city
                    j['_keyword'] = kw
                all_jobs.extend(results)
                print(f'{city} | {kw} | p{page} → {len(all_jobs)} total', end='\r')
                time.sleep(0.4)
            except Exception as e:
                print(f'Error {city}/{kw}/p{page}: {e}')

print(f'\nDone: {len(all_jobs)} raw postings')


Done: 2879 raw postings


In [20]:
ows, seen = [], set()

for j in all_jobs:
    jid = j.get('id', '')
    if jid in seen: continue
    seen.add(jid)
    rows.append({
        'id':          jid,
        'title':       j.get('title', ''),
        'company':     j.get('company', {}).get('display_name', ''),
        'description': j.get('description', ''),
        'salary_min':  j.get('salary_min', 0),
        'salary_max':  j.get('salary_max', 0),
        'city':        j.get('_city', ''),
        'search_title':j.get('_title', '')
    })

df = pd.DataFrame(rows)
df.to_csv('phase2_jobs_raw.csv', index=False)
print(f'Unique postings: {len(df)}')
print(df['city'].value_counts())

Unique postings: 5244
city
New York         1454
San Francisco    1354
Chicago          1252
Austin           1184
Name: count, dtype: int64


In [21]:
import duckdb

con = duckdb.connect()
con.register('jobs', df)

salary_by_city = con.execute("""
    SELECT
        city,
        COUNT(*) as postings,
        ROUND(AVG((salary_min + salary_max) / 2), 0) as avg_mid_salary
    FROM jobs
    WHERE salary_min > 20000 AND salary_max > 20000
    GROUP BY city
    ORDER BY avg_mid_salary DESC
""").df()

print(salary_by_city)

            city  postings  avg_mid_salary
0  San Francisco      1350        148193.0
1       New York      1446        139141.0
2         Austin      1178        121338.0
3        Chicago      1248        113877.0


In [22]:
title_volume = con.execute("""
    SELECT
        search_title,
        city,
        COUNT(*) as postings
    FROM jobs
    GROUP BY search_title, city
    ORDER BY postings DESC
    LIMIT 20
""").df()

print(title_volume)

  search_title           city  postings
0         None       New York       727
1                    New York       727
2         None  San Francisco       677
3               San Francisco       677
4         None        Chicago       626
5                     Chicago       626
6                      Austin       592
7         None         Austin       592


In [23]:
from collections import Counter

base_skills = ['sql','python','excel','tableau','power bi',
               'looker','snowflake','bigquery','dbt']

ai_skills   = ['chatgpt','copilot','llm','generative ai',
               'openai','prompt engineering','gpt','ai tools',
               'machine learning','artificial intelligence']

base_counts = Counter()
ai_counts   = Counter()
pairings    = Counter()

for desc in df['description'].dropna():
    dl = desc.lower()
    found_base = [s for s in base_skills if s in dl]
    found_ai   = [s for s in ai_skills   if s in dl]
    for s in found_base: base_counts[s] += 1
    for s in found_ai:   ai_counts[s]   += 1
    for b in found_base:
        for a in found_ai:
            pairings[f'{b} + {a}'] += 1

total = len(df)
print('BASE SKILLS')
for s,n in base_counts.most_common(9):
    print(f'  {s:20} {n:4} ({round(n/total*100)}%)')

print('\nAI SKILLS')
for s,n in ai_counts.most_common(10):
    print(f'  {s:20} {n:4} ({round(n/total*100)}%)')

print('\nTOP PAIRINGS (base + AI together)')
for s,n in pairings.most_common(10):
    print(f'  {s:30} {n:4} ({round(n/total*100)}%)')

BASE SKILLS
  excel                 258 (5%)
  sql                   152 (3%)
  python                 56 (1%)
  snowflake              34 (1%)
  tableau                34 (1%)
  power bi               34 (1%)
  dbt                     4 (0%)
  bigquery                2 (0%)
  looker                  2 (0%)

AI SKILLS
  machine learning      100 (2%)
  llm                    82 (2%)
  generative ai          48 (1%)
  artificial intelligence   42 (1%)
  openai                 26 (0%)
  chatgpt                 8 (0%)
  gpt                     8 (0%)
  copilot                 4 (0%)
  ai tools                4 (0%)

TOP PAIRINGS (base + AI together)
  python + machine learning         8 (0%)
  python + llm                      6 (0%)
  sql + machine learning            6 (0%)
  sql + llm                         4 (0%)
  excel + chatgpt                   2 (0%)
  excel + gpt                       2 (0%)
  snowflake + machine learning      2 (0%)
  tableau + machine learning        2 (0%)
 

**Limitations I flagged before presenting findings:**

1. Adzuna free tier caps descriptions at 500 characters.
   Skill % are understated — skills mentioned later in a job
   description get cut off. I noted this rather than presenting
   inflated numbers.

2. Deduplication by job ID only. Same role on multiple boards
   may still appear under different IDs.

3. "Remote" returns national results — salary comparisons
   between Remote and cities are directional only.

4. Sample per city/title combo is small (~50-150).
   City-level findings are trends, not statistically robust.

This section exists because calling out your own methodology. gaps is what a senior analyst does.

In [25]:
app_code = """
import streamlit as st
import pandas as pd
import plotly.express as px
import duckdb

st.set_page_config(page_title="Job Market Intelligence", layout="wide")
st.title("Job Market Intelligence — Phase 2")
st.caption("246+ real postings. 5 cities. 8 job title variants. Built by Nuha G.")

df = pd.read_csv("phase2_jobs_raw.csv")
con = duckdb.connect()
con.register("jobs", df)

city = st.selectbox("Filter by city", ["All"] + sorted(df["city"].unique().tolist()))
if city != "All":
    df = df[df["city"] == city]
    con.register("jobs", df)

st.subheader("Postings by job title")
fig1 = px.bar(
    df["search_title"].value_counts().reset_index(),
    x="count", y="search_title", orientation="h",
    labels={"count":"postings","search_title":""}
)
st.plotly_chart(fig1, use_container_width=True)

st.subheader("Salary by city")
sal = con.execute(
    "SELECT city, ROUND(AVG((salary_min+salary_max)/2),0) as avg_salary "
    "FROM jobs WHERE salary_min>20000 AND salary_max>20000 GROUP BY city ORDER BY avg_salary DESC"
).df()
fig2 = px.bar(sal, x="city", y="avg_salary", labels={"avg_salary":"avg mid salary"})
st.plotly_chart(fig2, use_container_width=True)

st.subheader("About this project")
st.markdown(
    "Built to answer: what do companies actually require, and is AI showing up yet? "
    "Non-traditional background: Combat Medic, Ops Coordinator, BBA Information Systems. "
    "Domain expertise in healthcare and operations — two of the fastest-growing data sectors. "
    "Limitations are documented in the notebook."
)
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("app.py created")

app.py created
